# NHL Fight Momentum — Data Extraction

This notebook identifies every confirmed fight in the NHL play-by-play dataset (2010–2024) and builds a before/after window around each one to measure momentum shifts. The output is a clean CSV ready for analysis.

**Fight identification logic:** A fight is defined as two teams both receiving a 5-minute major penalty at the exact same game time in the same game. This filters out solo major penalties (boarding, charging, spearing, etc.) which only appear on one team.

**Momentum metric:** Corsi (shot attempt differential) and shots on goal in the 5 minutes before vs. 5 minutes after the fight, measured from the perspective of each team involved.

In [ ]:
import pandas as pd
import numpy as np

# Load the raw play-by-play data
df = pd.read_csv('Play-By-Play_NHL.csv', low_memory=False)

# Filter to regular season games only (positions 4-6 of GameID = '02')
df = df[df['GameID'].astype(str).str[4:6] == '02'].copy()

# Add a Season column for later analysis
df['Season'] = df['GameID'].astype(str).str[:4].astype(int)

# Sort by game and event order
df = df.sort_values(['GameID', 'EventIndex']).reset_index(drop=True)

print(f'Total regular season rows loaded: {len(df):,}')
print(f'Seasons covered: {df["Season"].min()} to {df["Season"].max()}')

In [ ]:
# --- STEP 1: IDENTIFY FIGHTS ---

# Isolate all 5-minute major penalties
majors = df[
    (df['typeCode'] == 509) &
    (df['typeCode2'] == 'MAJ') &
    (df['PEN_duration'] == 5)
].copy()

# A fight = both teams receive a 5-min major at the same gameTime in the same game
fight_pairs = majors.groupby(['GameID', 'gameTime']).filter(lambda x: len(x) >= 2)

# Get one row per fight (the unique GameID + gameTime combinations)
fights = fight_pairs.groupby(['GameID', 'gameTime']).agg(
    Team_A=('EventTeam', lambda x: x.iloc[0]),
    Team_B=('EventTeam', lambda x: x.iloc[1]),
    period=('period', 'first'),
    Season=('Season', 'first')
).reset_index()

print(f'Total confirmed fights identified: {len(fights):,}')
print(f'Fights per season (sample):')
print(fights.groupby('Season').size())

In [ ]:
# --- STEP 2: BUILD BEFORE/AFTER WINDOWS ---

# For each fight, measure Corsi and shots in the 5 minutes (300 seconds)
# before and after the fight, for each team involved

WINDOW = 300  # seconds

def get_window_stats(game_df, fight_time, team):
    """
    For a given team in a given game, calculate Corsi and shots
    in the before and after windows around the fight.
    """
    # Before window: events in the 300 seconds leading up to the fight
    before = game_df[
        (game_df['gameTime'] >= fight_time - WINDOW) &
        (game_df['gameTime'] < fight_time) &
        (game_df['EventTeam'] == team)
    ]

    # After window: events in the 300 seconds following the fight
    after = game_df[
        (game_df['gameTime'] > fight_time) &
        (game_df['gameTime'] <= fight_time + WINDOW) &
        (game_df['EventTeam'] == team)
    ]

    return pd.Series({
        'Corsi_Before': before['Corsi'].sum(),
        'Corsi_After': after['Corsi'].sum(),
        'Shots_Before': before['Shot'].sum(),
        'Shots_After': after['Shot'].sum(),
        'Goals_Before': before['Goal'].sum(),
        'Goals_After': after['Goal'].sum(),
        'xG_Before': before['xG_F'].sum(),
        'xG_After': after['xG_F'].sum(),
        'Events_Before': len(before),
        'Events_After': len(after)
    })


results = []

for _, fight in fights.iterrows():
    game_df = df[df['GameID'] == fight['GameID']]

    for team_col in ['Team_A', 'Team_B']:
        team = fight[team_col]
        opponent = fight['Team_B'] if team_col == 'Team_A' else fight['Team_A']

        stats = get_window_stats(game_df, fight['gameTime'], team)

        # Get home/away status for the team
        team_rows = game_df[game_df['EventTeam'] == team]
        venue = team_rows['Venue'].iloc[0] if len(team_rows) > 0 else None

        # Get score state at the time of the fight
        fight_row = game_df[
            (game_df['gameTime'] == fight['gameTime']) &
            (game_df['EventTeam'] == team)
        ]
        score_state = fight_row['ScoreState'].iloc[0] if len(fight_row) > 0 else None

        results.append({
            'GameID': fight['GameID'],
            'Season': fight['Season'],
            'Fight_Time': fight['gameTime'],
            'Period': fight['period'],
            'Team': team,
            'Opponent': opponent,
            'Venue': venue,
            'Score_State_At_Fight': score_state,
            **stats
        })

fight_df = pd.DataFrame(results)

print(f'Rows in output (2 per fight, one per team): {len(fight_df):,}')
print(fight_df.head())

In [ ]:
# --- STEP 3: ENGINEER FINAL FEATURES ---

# Momentum delta: positive = team improved after the fight
fight_df['Corsi_Delta'] = fight_df['Corsi_After'] - fight_df['Corsi_Before']
fight_df['Shots_Delta'] = fight_df['Shots_After'] - fight_df['Shots_Before']
fight_df['xG_Delta'] = fight_df['xG_After'] - fight_df['xG_Before']

# Binary momentum flag: did the team improve in shot attempts after the fight?
fight_df['Gained_Momentum'] = (fight_df['Corsi_Delta'] > 0).astype(int)

# Is the team at home?
fight_df['Is_Home'] = (fight_df['Venue'] == 'Home').astype(int)

# Was the team losing at the time of the fight?
fight_df['Was_Trailing'] = (fight_df['Score_State_At_Fight'] < 0).astype(int)

# Was the fight early (period 1), middle (period 2), or late (period 3)?
fight_df['Period_Label'] = fight_df['Period'].map({1: 'Period 1', 2: 'Period 2', 3: 'Period 3'})

# Drop rows where windows were too short (near start/end of period)
fight_df = fight_df[
    (fight_df['Events_Before'] > 0) &
    (fight_df['Events_After'] > 0)
].copy()

print(f'Clean rows after filtering: {len(fight_df):,}')
print()
print(fight_df[['GameID','Season','Team','Period','Corsi_Before','Corsi_After',
               'Corsi_Delta','Shots_Delta','xG_Delta','Gained_Momentum']].head(10).to_string())

In [ ]:
# --- STEP 4: EXPORT ---

final_cols = [
    'GameID', 'Season', 'Fight_Time', 'Period', 'Period_Label',
    'Team', 'Opponent', 'Is_Home', 'Score_State_At_Fight', 'Was_Trailing',
    'Corsi_Before', 'Corsi_After', 'Corsi_Delta',
    'Shots_Before', 'Shots_After', 'Shots_Delta',
    'Goals_Before', 'Goals_After',
    'xG_Before', 'xG_After', 'xG_Delta',
    'Gained_Momentum'
]

fight_df[final_cols].to_csv('NHL_Fight_Momentum_Data.csv', index=False)

print('Export complete: NHL_Fight_Momentum_Data.csv')
print()
print('Dataset summary:')
print(fight_df[final_cols].describe())